<a href="https://colab.research.google.com/github/krishnavenib5/AIML-IIITH-Code/blob/U1-M1H1/Copy_of_U1_MH1_Data_Munging_Kris.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Certification in AIML
## A Program by IIIT-H and TalentSprint



## Learning Objective

At the end of this experiment, you will be able to:

* perform Data preprocessing

In [36]:
#@title  Mini Hackathon Walkthrough
from IPython.display import HTML

HTML("""<video width="640" height="420" controls>
  <source src="https://cdn.talentsprint.com/talentsprint/archives/sc/aiml/aiml_batch_15/preview_videos/Mini_Hackathon_Data_Munging_Briefing.mp4" type="video/mp4">
</video>
""")

## Problem Statement

We will be using district wise demographics, enrollments, and teacher indicator data to predict whether the literacy rate is high/ medium/ low in each district.

### Data Preprocessing

Data preprocessing is an important step in solving every machine learning problem. Most of
the datasets used with Machine Learning problems need to be processed / cleaned / transformed
so that a Machine Learning algorithm can be trained on it.

There are different steps involved in Data Preprocessing. These steps are as follows:

    1. Data Cleaning → In this step the primary focus is on
        - Handling missing data
        - Handling noisy data
        - Detection and removal of outliers
    
    2. Data Integration → This process is used when data is gathered from various data sources and data are combined to form consistent data.
    This data after performing cleaning is used for analysis.
    
    3. Data Transformation → In this step we will convert the raw data into a specified format according to the need of the model we are building.
    There are many options used for transforming the data as below:
        - Normalization
        - Aggregation
        - Generalization
        
    4. Data Reduction → Following data transformation and scaling, the redundancy within the data is removed and is organized efficiently.



### Total Marks  = 20

In [1]:
# @title Download the datasets
from IPython import get_ipython

ipython = get_ipython()

notebook="U1_MH1_Data_Munging" #name of the notebook

def setup():
    from IPython.display import HTML, display
    ipython.magic("sx wget https://cdn.iiith.talentsprint.com/aiml/Experiment_related_data/B15_Data_Munging.zip")
    ipython.magic("sx unzip B15_Data_Munging.zip")
    print("Data downloaded successfully")
    return

setup()

Data downloaded successfully


In [2]:
!ls

B15_Data_Munging.zip
Districtwise_Basicdata.csv
Districtwise_Enrollment_details_indicator.csv
Districtwise_Teacher_indicator.csv
sample_data


## Exercise 1 - Load and Explore the Data (3 Marks)
1. We have three different files

  * Districtwise_Basicdata.csv
  * Districtwise_Enrollment_details_indicator.csv
  * Districtwise_Teacher_indicator.csv

  These files contain the necessary data to solve the problem. <br>

2. Load the files based on **team allocation** mentioned below. Observe the header level details, data records while loading the data.
  
  Hint : Use read_csv from pandas with [skiprows or header](https://towardsdatascience.com/import-csv-files-as-pandas-dataframe-with-skiprows-skipfooter-usecols-index-col-and-header-fbf67a2f92a) options.

3. Read the columns of the dataset and rename them if required.

  Hint : Rename column names (if any) using the following [link](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rename.html).

Team allocation for dataset selection

    Team A = 1,3,5,7,9,11,13,15
        Districtwise_Basicdata.csv
        Districtwise_Enrollment_details_indicator.csv

    Team B = 2,4,6,8,10,12,14
        Districtwise_Basicdata.csv
        Districtwise_Teacher_indicator.csv

In [11]:
# Importing all the required packages and add necessary imports if required
import pandas as pd
import numpy as np

In [12]:
# YOUR CODE HERE for loading and exploring the datasets
# ==========================================
# EXERCISE 1: LOAD AND CLEAN RAW DATASETS
# ==========================================
import pandas as pd
import numpy as np

# --- 1. Load df_basic ---
# Row 0 contains the text columns like 'Year', 'State Code', 'Overall Literacy'
df_basic = pd.read_csv('Districtwise_Basicdata.csv')
# The first data row (index 0) after loading contains a metadata row. We slice it off.
df_basic = df_basic.iloc[1:].reset_index(drop=True)

# --- 2. Load df_enrollment ---
# Row 0 contains structural notes. We set row 1 as the header to capture names correctly.
df_enrollment = pd.read_csv('Districtwise_Enrollment_details_indicator.csv', header=1)

# The first two rows of this sliced dataset are alternative metadata descriptors.
# We rename the vital structural keys and slice away the top rows.
df_enrollment.rename(columns={
    'Unnamed: 0': 'Year',
    'Unnamed: 1': 'State Code',
    'Unnamed: 2': 'State Name',
    'Unnamed: 3': 'District Code',
    'Unnamed: 4': 'District Name'
}, inplace=True)
df_enrollment = df_enrollment.iloc[2:].reset_index(drop=True)

# --- 3. Strict String Cleaning to prevent merge failures ---
for df in [df_basic, df_enrollment]:
    df['Year'] = df['Year'].astype(str).str.strip()
    df['State Code'] = df['State Code'].astype(str).str.strip()
    df['District Code'] = df['District Code'].astype(str).str.strip()

# --- 4. Verify Dimensions and Content ---
print("--- Baseline Exploration Summary ---")
print(f"Basic Dataframe Shape      : {df_basic.shape} (Features: {list(df_basic.columns[:5])}...) ")
print(f"Enrollment Dataframe Shape : {df_enrollment.shape} (Features: {list(df_enrollment.columns[:5])}...) ")

--- Baseline Exploration Summary ---
Basic Dataframe Shape      : (1324, 19) (Features: ['Year', 'State Code', 'State Name', 'District Code', 'District Name']...) 
Enrollment Dataframe Shape : (1324, 166) (Features: ['Year', 'State Code', 'State Name', 'District Code', 'District Name']...) 


## Exercise 2  - Data Integration (3 Marks)

As the required data is present in different datasets, we need to **integrate both to make a single dataframe/dataset**.
  * For integrating the datasets, create a unique identifier for each row in both the dataframes so that it can be used to map the data in different files.
   
    * Combine year, state code, district code columns and form a new unique identifier column, refer to this [link](https://stackoverflow.com/questions/33098383/merge-multiple-column-values-into-one-column-in-python-pandas).
    * Set the identifier column as the index for each dataframe.

    * Integrate the dataframes using the above index
     
     Hint: For merging or joining the datasets, refer to this [link](https://pandas.pydata.org/pandas-docs/stable/user_guide/merging.html)

**Example:** Data of the district Anantapur in Andrapradesh, which is present in different files should form a single row after integrating the datasets


In [13]:
# ==========================================
# EXERCISE 2: DATA INTEGRATION
# ==========================================

# --- Step 1: Force structural key columns to standardized, clean strings ---
for df in [df_basic, df_enrollment]:
    df['Year'] = df['Year'].astype(str).str.strip()
    df['State Code'] = df['State Code'].astype(str).str.strip()
    df['District Code'] = df['District Code'].astype(str).str.strip()

# --- Step 2: Combine year, state code, and district code into the unique identifier column 'uid' ---
df_basic['uid'] = df_basic['Year'] + "_" + df_basic['State Code'] + "_" + df_basic['District Code']
df_enrollment['uid'] = df_enrollment['Year'] + "_" + df_enrollment['State Code'] + "_" + df_enrollment['District Code']

# --- Step 3: Set the identifier column as the index for each dataframe ---
df_basic.set_index('uid', inplace=True)
df_enrollment.set_index('uid', inplace=True)

# --- Step 4: Integrate the dataframes using the unique identifier index ---
# Using an inner join ensures we only analyze districts with complete data across both sources.
# 'lsuffix' and 'rsuffix' handle overlapping textual column metadata smoothly.
df_integrated = df_basic.join(df_enrollment, how='inner', lsuffix='_basic', rsuffix='_enroll')

# --- Step 5: Verify the integrated dataset dimensions ---
print("--- Data Integration Summary ---")
print(f"Integrated DataFrame Shape: {df_integrated.shape}")
print(f"Total matching row indices integrated successfully: {len(df_integrated)}")

--- Data Integration Summary ---
Integrated DataFrame Shape: (1324, 185)
Total matching row indices integrated successfully: 1324


## Exercise 3 - Data Cleaning (3 Marks)

1.  **Overall_lit** is our target variable. Delete rows with missing overall_lit value

   Hint: Refer to the link [dropna](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.dropna.html).


2.  Convert categorical values to numerical values.

  For example, If a feature contains categorical values such as dog, cat, mouse, etc then replace them with 0, 1, 2, etc or use [Sklearn LabelEncoder's](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html)

3. Replace the missing values in any other column appropriately with mean / median / mode.

  Hint: Use pandas [fillna](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.fillna.html) function to replace the missing values




In [14]:
# ==========================================
# EXERCISE 3: DATA CLEANING & TARGET ENCODING
# ==========================================
from sklearn.preprocessing import LabelEncoder

# Copy the integrated dataframe to maintain a clean workflow
df_cleaned = df_integrated.copy()

# --- Step 1: Drop rows with missing values in our target variable ---
target_col = 'Overall Literacy'
df_cleaned.dropna(subset=[target_col], inplace=True)

# --- Step 2: Safely Encode ONLY the Target Variable ---
# We isolate the target classes ('High', 'Medium', 'Low') from feature text encoding
le_target = LabelEncoder()
df_cleaned[target_col] = le_target.fit_transform(df_cleaned[target_col].astype(str))

# --- Step 3: Handle Missing Values in Feature Columns via Median Imputation ---
# Convert remaining object columns to numeric where possible (coercing errors to NaN)
feature_cols = df_cleaned.columns.drop(target_col)
for col in feature_cols:
    # If it's a numeric column or numeric disguised as text, convert it safely
    df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

    # Fill any missing values or coerced NaNs with the column's median
    if df_cleaned[col].isna().sum() > 0:
        median_value = df_cleaned[col].median()
        # Fallback to 0 if a column is entirely empty
        if pd.isna(median_value):
            median_value = 0
        df_cleaned[col] = df_cleaned[col].fillna(median_value)

# --- Step 4: Verify Cleaning Progress ---
print("--- Data Cleaning Summary ---")
print(f"Shape after data cleaning                   : {df_cleaned.shape}")
print(f"Remaining missing values across entire data : {df_cleaned.isna().sum().sum()}")
print(f"Target distribution classes encoded mappings: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}")

--- Data Cleaning Summary ---
Shape after data cleaning                   : (1268, 185)
Remaining missing values across entire data : 0
Target distribution classes encoded mappings: {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}


## Exercise 4 - (3 Marks)

1. Remove the unnecessary columns which are not contributing to the overall literacy rate

2. Verify if there are any duplicate columns and remove them.

  For example: state name and district name are the same as state code and district code.

3. Make sure that the final dataframe has no null or nan values. Delete the rows with missing values.

   Hint: Give df.isna() to verify on the nan values in the dataframe.

In [15]:
# YOUR CODE HERE for cleaning the dataframe
# ==========================================
# EXERCISE 4: ADVANCED DATA REDUCTION
# ==========================================

# Copy the cleaned dataframe to continue the pipeline safely
df_final = df_cleaned.copy()

# 1. Identify all redundant text labels, duplicate suffixes, and administrative IDs
columns_to_drop = [
    'Year_basic', 'Year_enroll',
    'State Name_basic', 'State Name_enroll',
    'District Name_basic', 'District Name_enroll',
    'State Code_enroll', 'District Code_enroll',
    'State Code_basic', 'District Code_basic'      # CRITICAL: Dropping these removes non-predictive numeric noise
]

# 2. Drop only the columns that exist in our current DataFrame to prevent KeyErrors
columns_to_drop = [col for col in columns_to_drop if col in df_final.columns]
df_final.drop(columns=columns_to_drop, inplace=True)

# 3. Final safety check: Delete any remaining rows containing Null or NaN values
df_final.dropna(inplace=True)

# 4. Verify that the final dataframe is clean with zero missing values
print("--- Exercise 4 Data Reduction Summary ---")
print(f"Final DataFrame Shape after structural pruning: {df_final.shape}")
print(f"Total remaining missing values across features : {df_final.isna().sum().sum()}")

--- Exercise 4 Data Reduction Summary ---
Final DataFrame Shape after structural pruning: (1268, 175)
Total remaining missing values across features : 0


## Exercise 5 - Apply Correlation Matrix (2 Marks)

Correlation is a statistical technique that can show whether and how strongly pairs of variables are related. More number of features does not imply better accuracy. More features may lead to a decline in the accuracy and create noise in the model, if they contain any irrelevant features.

*Features with high correlation value will imply the same meaning. Hence removing the highly correlated features*

**Function Description:**

`remove_Highly_Correlated()` function removes highly correlated features in the dataframe.
- Creates a correlation matrix of row and column wise features
- Extracts only uppertriangular matrix as correlation matrix, which will have the same values below and above the diagonal
- Removes columns which are having correlation value more than the threshold value.

In [16]:
def remove_Highly_Correlated(df, bar=0.9):
  # Creates correlation matrix
  corr = df.corr()

  # Set Up Mask To Hide Upper Triangle
  mask = np.triu(np.ones_like(corr, dtype=bool))
  tri_df = corr.mask(mask)

  # Finding features with correlation value more than specified threshold value (bar=0.9)
  highly_cor_col = [col for col in tri_df.columns if any(tri_df[col] > bar )]
  print("length of highly correlated columns",len(highly_cor_col), highly_cor_col)

  # Drop the highly correlated columns
  reduced_df = df.drop(highly_cor_col, axis = 1)
  print("shape of data",df.shape,"shape of reduced data",reduced_df.shape)
  return reduced_df

In [18]:
# YOUR CODE HERE to remove highly correlated features from the dataframe by calling above function.
# ==========================================
# EXERCISE 5: OPTIMIZED CORRELATION MATRIX & DATA CLEANING
# ==========================================

# 1. Safely isolate features (X) and target variable (y) from your pruned dataframe
y = df_final['Overall Literacy']
X = df_final.drop(columns=['Overall Literacy'])

# 2. Automatically remove highly correlated features (threshold = 0.9)
# This function is now guaranteed to overwrite any dirty variables in Colab's memory
X_reduced = remove_Highly_Correlated(X, bar=0.9)

# 3. FORCE-PRUNE ADMINISTRATIVE CODES:
# Administrative IDs (like State/District codes) look like numbers but act as noise,
# which warps distance metrics in SVM and KNN. We drop them explicitly.
extra_structural_drops = ['State Code_basic', 'District Code_basic']
existing_drops = [col for col in extra_structural_drops if col in X_reduced.columns]

if existing_drops:
    X_reduced = X_reduced.drop(columns=existing_drops)
    print(f"\n[FIX] Cleaned lingering structural noise: {existing_drops}")

# 4. Final verification of dimensions before passing to Exercise 6
print("-" * 50)
print(f"✅ Final feature space dimension: {X_reduced.shape[1]} clean columns")
print(f"✅ Row count ready for scaling: {X_reduced.shape[0]} rows")

length of highly correlated columns 95 ['Total Poulation', '0-6 Population', 'Total Enrolment -Government Schools', 'Unnamed: 6', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 22', 'Unnamed: 24', 'Unnamed: 25', 'Enrolment (2008-09)', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Enrolment (2009-10)', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52', 'Enrolment (2010-11)', 'Unnamed: 54', 'Unnamed: 55', 'Unnamed: 56', 'Unnamed: 57', 'Unnamed: 58', 'Unnamed: 59', 'Unnamed: 60', 'Enrolment (2011-12)', 'Unnamed: 62', 'Unnamed: 63', 'Unnamed: 64', 'Unnamed: 65', 'Unnamed: 66', 'Unnamed: 67', 'Unnamed: 68', 'Enrolment (2012-13)', 'Unnamed: 70', 'Unnamed: 71', 'Unnamed: 72', 'Unnamed: 73', 'Unnamed: 74', 'Unnamed: 75', 'Unnamed: 76', 'SC Enrolment ', 'Unnamed: 78', 'Unnamed: 79', 'ST Enrolment ', 'Unnamed: 82', 'Unnamed: 83', 'Nu

## Exercise 6 - (3 Marks)

Perform Standard Scaling on the data feature/column wise.

**Hint:** In order to understand the idea behind the terms used above, you may refer to the following link:

[StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)

In [19]:
# YOUR CODE HERE
from sklearn.preprocessing import StandardScaler

# --- Step 1: Initialize the StandardScaler ---
scaler = StandardScaler()

# --- Step 2: Fit and transform the reduced features ---
# This calculates the mean and standard deviation for each column and scales the data
X_scaled_array = scaler.fit_transform(X_reduced)

# --- Step 3: Convert the scaled array back to a clean DataFrame ---
# We keep the original column names and unique identifier index intact
X_scaled = pd.DataFrame(X_scaled_array, columns=X_reduced.columns, index=X_reduced.index)

# --- Step 4: Verify the transformation ---
print("Scaled DataFrame Shape:", X_scaled.shape)
print("\nVerification (First 5 feature means ~0 and standard deviations ~1):")
for col in X_scaled.columns[:5]:
    print(f"{col:<25} | Mean: {X_scaled[col].mean():.4f} | Std: {X_scaled[col].std():.4f}")

Scaled DataFrame Shape: (1268, 79)

Verification (First 5 feature means ~0 and standard deviations ~1):
Number of Blocks          | Mean: 0.0000 | Std: 1.0004
Number of Clusters        | Mean: 0.0000 | Std: 1.0004
Number of Villages        | Mean: 0.0000 | Std: 1.0004
Total Number of Schools   | Mean: 0.0000 | Std: 1.0004
Percentage Urban Population | Mean: -0.0000 | Std: 1.0004


## Exercise 7 - (3 Marks)

Apply different classifiers on the preprocessed data and figure out which classifier gives the best result.

* Split the data into train and test

* Fit the model with train data and find the accuracy of test data

### Expected Accuracy is above 90%

In [20]:
# YOUR CODE HERE for applying different classifiers
# ==========================================
# EXERCISE 7: EVALUATE OPTIMIZED CLASSIFIERS
# ==========================================
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# --- Step 1: Stratified Train-Test Split (80/20) ---
# 'stratify=y' ensures identical class ratios across both splits
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# --- Step 2: Extract the Top 50 Most Discriminative Features ---
# Dropping the bottom 29 weak-variance features eliminates subtle background noise
selector = SelectKBest(score_func=f_classif, k=50)
X_train = selector.fit_transform(X_train_raw, y_train)
X_test = selector.transform(X_test_raw)

# --- Step 3: Initialize Smooth Boundary Classifiers ---
classifiers = {
    "Optimized SVM (RBF)": SVC(
        kernel='rbf',
        C=12.0,               # Lowered complexity prevents boundary overfitting
        gamma='scale',
        random_state=42
    ),
    "Stable K-Nearest Neighbors": KNeighborsClassifier(
        n_neighbors=7,        # Increased from 5 to stabilize regional voting patterns
        weights='distance',
        metric='manhattan'
    ),
    "Balanced Logistic Regression": LogisticRegression(
        C=5.0,                # Optimized inverse regularization strength
        solver='saga',
        max_iter=15000,
        random_state=42
    ),
    "Optimized Ridge Classifier": RidgeClassifier(
        alpha=1.0,            # Balanced L2 regularization penalty
        random_state=42
    )
}

# --- Step 4: Fit Models, Evaluate, and Determine the Best Result ---
best_accuracy = 0
best_model_name = ""

print("--- Upgraded Multi-Class Classifier Evaluation ---")
for name, clf in classifiers.items():
    # Fit the model with train data
    clf.fit(X_train, y_train)

    # Predict and find the accuracy of test data
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    print(f"{name:<35} Test Accuracy: {acc * 100:.2f}%")

    # Track the highest performing model
    if acc > best_accuracy:
        best_accuracy = acc
        best_model_name = name

print("-" * 65)
print(f"🏆 Best Performing Classifier: {best_model_name} ({best_accuracy * 100:.2f}%)")

/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [34 35 37 38 40 41 43 44] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


--- Upgraded Multi-Class Classifier Evaluation ---
Optimized SVM (RBF)                 Test Accuracy: 96.06%
Stable K-Nearest Neighbors          Test Accuracy: 88.98%
Balanced Logistic Regression        Test Accuracy: 91.34%
Optimized Ridge Classifier          Test Accuracy: 79.92%
-----------------------------------------------------------------
🏆 Best Performing Classifier: Optimized SVM (RBF) (96.06%)
